# QUBO Diagnostics Notebook

This notebook is an interactive front-end for the Steiner Tree QUBO diagnostics code. It reuses the existing formulation builders and the diagnostics helpers in `SteinerTreeProblemQUBO/diagnostics/qubo_diagnostics.py`, while letting you run baseline comparisons, penalty sweeps, ablations, and CSV exports cell by cell.

In [1]:
from pathlib import Path
import sys
from datetime import datetime

def find_repo_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "SteinerTreeProblemQUBO").is_dir():
            return candidate
    raise RuntimeError("Could not find repo root containing SteinerTreeProblemQUBO")

REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from SteinerTreeProblemQUBO.diagnostics import qubo_diagnostics as diag

REPO_ROOT

PosixPath('/Users/daghanerdonmez/Desktop/evvifing/boun/cmpe/491-492/finale')

## Configuration

Edit the problem generator, penalty weights, read counts, and output location here.

In [2]:
# PROBLEM_SPEC = {
#     "generator": "sparsity",
#     "params": {
#         "node_count": 10,
#         "terminal_count": 4,
#         "extra_edge_probability": 0.3,
#         "weight_range": (1, 100),
#         "seed": 0,
#     },
# }

#Other examples:
PROBLEM_SPEC = {
    "generator": "geometric",
    "params": {
        "node_count": 20,
        "terminal_count": 5,
        "connectivity": "knn",
        "k": 5,
        "max_weight": 100,
        "seed": 0,
    },
}

BASE_CONSTRAINT_WEIGHT = None
TOP_K_SAMPLES = 10
NUM_READS = 100
PENALTY_GRID = [1, 3, 10, 30, 100, 300, 1000]
SQA_KWARGS = {
    "num_sweeps": 4000,
    "trotter": 16,
}
TRY_GUROBI_FIRST = True
EXACT_SOLVER_NODE_LIMIT = 16

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_DIR = REPO_ROOT / "SteinerTreeProblemQUBO" / "logs" / f"qubo_diagnostics_notebook_{timestamp}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_DIR

PosixPath('/Users/daghanerdonmez/Desktop/evvifing/boun/cmpe/491-492/finale/SteinerTreeProblemQUBO/logs/qubo_diagnostics_notebook_20260518_151103')

## Problem Setup

In [3]:
problem = diag.make_problem(PROBLEM_SPEC)
constraint_weight = (
    float(BASE_CONSTRAINT_WEIGHT)
    if BASE_CONSTRAINT_WEIGHT is not None
    else diag.default_constraint_weight(problem)
)
optimal_cost, optimal_source = diag.compute_optimal_cost(problem)

diag.print_problem_summary(problem, optimal_cost, optimal_source, constraint_weight)

Set parameter Username
Set parameter LicenseID to value 2801207
Academic license - for non-commercial use only - expires 2027-04-01
nodes=20 edges=57 terminals=5
root=v0 terminals=['v0', 'v15', 'v10', 'v7', 'v17']
constraint_weight=501.0
optimal_cost=410.0 source=gurobi



In [4]:
sample_rows = []
coefficient_rows = []
penalty_rows = []
ablation_rows = []
baseline_results = {}

## Baseline Diagnostics

Run this cell to compare `hybrid` and `alex` on the same instance. It prints coefficient-scale diagnostics, the top unique OpenJij samples, per-term energies, decoded edges, and violation details.

In [5]:
baseline_results = {}

for model_name in ("hybrid", "alex"):
    result = diag.run_model_diagnostics(
        problem,
        model_name=model_name,
        constraint_weight=constraint_weight,
        num_reads=NUM_READS,
        top_k=TOP_K_SAMPLES,
        experiment="baseline",
        optimal_cost=optimal_cost,
        coefficient_rows=coefficient_rows,
        sample_rows=sample_rows,
    )
    baseline_results[model_name] = result
    diag.print_top_sample_summary(model_name, result["top_analyses"], result["top_summary"])

[baseline:hybrid] coeffs vars=871 quads=11882 lin=[-2084160.0, 2164320.0] quad=[-513024.0, 2052096.0] maxabs=2164320.0 offset=51999792.0
baseline:hybrid: 100%|██████████| 100/100 [01:37<00:00,  1.02read/s]
[hybrid] top-10 unique samples
  rank=1 occ=1 energy=24913.000 class=infeasible tree_cost=865.000 active_bits=381
    terms: H_cost=865.000 | H_terminal_parent=5010.000 | H_nonterminal_parent=10020.000 | H_no_fake_root=1503.000 | H_root_depth=0.000 | H_depth=7515.000
    edges: v0-v8(59.0); v0-v9(38.0); v10-v13(44.0); v10-v15(54.0); v4-v10(56.0); v15-v16(56.0); v3-v17(95.0); v7-v17(64.0); v7-v18(46.0); v2-v7(49.0); v3-v5(62.0); v4-v15(72.0); v8-v11(77.0); v9-v11(62.0); v9-v14(31.0)
    violations: terminal_not_reached[4.0]: Unreachable terminals from root v0: ['v10', 'v15', 'v17', 'v7'] | detached_component[11.0]: Selected nodes not reachable from root v0: ['v10', 'v13', 'v15', 'v16', 'v17', 'v18', 'v2', 'v3', 'v4', 'v5', 'v7'] | terminal_parent[1.0]: Terminal v17 has indegree 0, exp

In [ ]:
# Peek directly at the top analyzed samples if you want structured access.
baseline_results["hybrid"]["top_analyses"][:2]

## Penalty Sweep

This runs the hybrid model over `PENALTY_GRID` and records feasible rate, best feasible tree cost, best infeasible energy, and the most common violation type.

In [ ]:
penalty_rows = []

for penalty in PENALTY_GRID:
    result = diag.run_model_diagnostics(
        problem,
        model_name="hybrid",
        constraint_weight=float(penalty),
        num_reads=NUM_READS,
        top_k=TOP_K_SAMPLES,
        experiment="penalty_sweep",
        optimal_cost=optimal_cost,
        coefficient_rows=coefficient_rows,
        sample_rows=sample_rows,
    )
    summary = result["all_summary"]
    row = {
        "constraint_weight": float(penalty),
        "best_total_energy": result["aggregated"][0]["energy_without_offset"] + result["bundle"].labeled_bqm.offset,
        "num_feasible_unique": summary["num_feasible"],
        "num_infeasible_unique": summary["num_infeasible"],
        "best_feasible_energy": summary["best_feasible_energy"],
        "best_infeasible_energy": summary["best_infeasible_energy"],
        "best_feasible_tree_cost": summary["best_decoded_tree_cost"],
        "feasible_sample_rate": summary["feasible_sample_rate"],
        "optimal_hit_rate": summary["optimal_hit_rate"],
        "most_common_violation_type": summary["most_common_violation_type"],
        "violation_frequency": diag._flatten_violation_frequency(summary["violation_frequency"]),
    }
    penalty_rows.append(row)
    print(
        f"penalty={penalty:<5} "
        f"best_total_energy={row['best_total_energy']:.3f} "
        f"best_feasible_tree_cost={row['best_feasible_tree_cost']} "
        f"feasible_rate={row['feasible_sample_rate']:.3f} "
        f"optimal_hit_rate={row['optimal_hit_rate']} "
        f"common_violation={row['most_common_violation_type'] or '(none)'}"
    )

In [ ]:
penalty_rows

## Ablation Study

This runs the hybrid model with terms added incrementally using the actual term names present in the current hybrid implementation.

In [ ]:
ablation_rows = []

for stage_name, enabled_terms in diag.HYBRID_ABLATIONS.items():
    result = diag.run_model_diagnostics(
        problem,
        model_name="hybrid",
        constraint_weight=constraint_weight,
        num_reads=NUM_READS,
        top_k=TOP_K_SAMPLES,
        experiment="ablation",
        optimal_cost=optimal_cost,
        coefficient_rows=coefficient_rows,
        sample_rows=sample_rows,
        ablation_stage=stage_name,
        enabled_terms=enabled_terms,
    )
    summary = result["all_summary"]
    row = {
        "ablation_stage": stage_name,
        "enabled_terms": "; ".join(enabled_terms),
        "constraint_weight": constraint_weight,
        "best_total_energy": result["aggregated"][0]["energy_without_offset"] + result["bundle"].labeled_bqm.offset,
        "num_feasible_unique": summary["num_feasible"],
        "num_infeasible_unique": summary["num_infeasible"],
        "best_feasible_energy": summary["best_feasible_energy"],
        "best_infeasible_energy": summary["best_infeasible_energy"],
        "best_feasible_tree_cost": summary["best_decoded_tree_cost"],
        "feasible_sample_rate": summary["feasible_sample_rate"],
        "optimal_hit_rate": summary["optimal_hit_rate"],
        "most_common_violation_type": summary["most_common_violation_type"],
        "violation_frequency": diag._flatten_violation_frequency(summary["violation_frequency"]),
    }
    ablation_rows.append(row)
    print(
        f"stage={stage_name:<24} "
        f"best_total_energy={row['best_total_energy']:.3f} "
        f"best_feasible_tree_cost={row['best_feasible_tree_cost']} "
        f"feasible_rate={row['feasible_sample_rate']:.3f} "
        f"common_violation={row['most_common_violation_type'] or '(none)'}"
    )

In [ ]:
ablation_rows

## Coefficient Diagnostics

In [ ]:
coefficient_rows[-10:]

## Export CSVs

In [ ]:
diag.write_csv(OUTPUT_DIR / "sample_diagnostics.csv", sample_rows)
diag.write_csv(OUTPUT_DIR / "penalty_sweep.csv", penalty_rows)
diag.write_csv(OUTPUT_DIR / "ablation_results.csv", ablation_rows)
diag.write_csv(OUTPUT_DIR / "coefficient_stats.csv", coefficient_rows)

print(OUTPUT_DIR / "sample_diagnostics.csv")
print(OUTPUT_DIR / "penalty_sweep.csv")
print(OUTPUT_DIR / "ablation_results.csv")
print(OUTPUT_DIR / "coefficient_stats.csv")

## Optional Reset

Run this if you want to restart the notebook session without reloading the kernel.

In [ ]:
sample_rows = []
coefficient_rows = []
penalty_rows = []
ablation_rows = []
baseline_results = {}
print("Cleared in-memory diagnostic results.")